# Capstone — HNSW vs IVF vs brute force

Generate clustered synthetic vectors, build both indexes, plot recall@10 vs query latency.

This notebook is the only place where matplotlib is required.

In [ ]:
import sys, os, numpy as np
ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if ROOT not in sys.path: sys.path.insert(0, ROOT)
MODULE = os.path.abspath(os.path.join(os.getcwd()))
if MODULE not in sys.path: sys.path.insert(0, MODULE)
from ann.hnsw import HNSW
from ann.ivf import IVF
from ann.bench import brute_force_topk, gen_clustered_vectors, recall_at_k
from common.bench import time_it

In [ ]:
N, D, K, NQ = 5000, 32, 10, 50
rng = np.random.default_rng(0)
corpus = gen_clustered_vectors(N, D, n_clusters=24, seed=1)
queries = rng.normal(scale=2.0, size=(NQ, D)).astype(np.float32)
truths = [brute_force_topk(q, corpus, k=K) for q in queries]

In [ ]:
h = HNSW(dim=D, M=16, ef_construction=200, ef_search=64, seed=0)
for v in corpus: h.add(v)
preds = [[nid for nid, _ in h.query(q, k=K)] for q in queries]
print('HNSW recall@10:', np.mean([recall_at_k(p, t) for p, t in zip(preds, truths)]))
r = time_it(lambda: [h.query(q, k=K) for q in queries], repeat=3, label='hnsw')
print('HNSW total query (ms):', r.best_ms)

In [ ]:
iv = IVF(nlist=64, n_iter=15, seed=0)
iv.fit(corpus)
preds = [[nid for nid, _ in iv.query(q, k=K, nprobe=8)] for q in queries]
print('IVF recall@10:', np.mean([recall_at_k(p, t) for p, t in zip(preds, truths)]))
r = time_it(lambda: [iv.query(q, k=K, nprobe=8) for q in queries], repeat=3, label='ivf')
print('IVF total query (ms):', r.best_ms)

In [ ]:
r = time_it(lambda: [brute_force_topk(q, corpus, k=K) for q in queries], repeat=3, label='brute')
print('Brute force total query (ms):', r.best_ms)

### Recall vs efSearch (HNSW)


In [ ]:
from common.visualize import plot_growth
ef_grid = [16, 32, 64, 128, 256, 512]
recalls = []
for ef in ef_grid:
    h.ef_search = ef
    preds = [[nid for nid, _ in h.query(q, k=K)] for q in queries]
    recalls.append(float(np.mean([recall_at_k(p, t) for p, t in zip(preds, truths)])))
plot_growth(ef_grid, {'recall@10': recalls}, title='HNSW recall vs efSearch', ylabel='recall', log=False);